# Stage D / NB 22 — reproducibility and reference audit

Protocol reference: **R1.6** (preprint citations), **R1.10** (language pass), **R2.c**
(open-weight reproducibility), and §9's release checklist.

## What this notebook is for

The last automatic gate before submission. It writes a last-built SHA-256 artifact manifest,
separates controlled working files from an explicit publication allowlist, audits structured
model identifiers and immutable revisions, reads text and Word manuscript references, preserves
the hand-written protocol-deviation log, and binds NB 17–21 through their gates and fingerprints.

## R2.c is answered by construction, not assertion

Every model in this revision is intended to be open-weight and locally hosted. The audit checks
structured model-ID fields rather than free-text notes, requires immutable registry revisions and
recorded seeds, and saves the environment. A passing file from another run cannot rescue stale
results because the release checklist checks the shared bootstrap, reference arm, final Holm
family counts, and qualitative-selection fingerprints.

## Reference handling

Preprints without a peer-reviewed DOI are marked `replace`; ambiguous DOI/preprint entries are
marked `verify`. Model cards may be cited as software artifacts, not clinical evidence, only when
the citation or pinned model registry supplies an immutable revision hash.

## Outputs (under `stage_D/nb22_release/`)
`release_checklist.csv/.md`, `artifact_manifest.csv`, `release_allowlist.csv`,
`restricted_data_hits.csv`, `restricted_working_hits.csv`, `model_references.csv`,
`reference_audit.csv`, `protocol_deviations.csv/.md`, `environment.json`, `run_config.json`, and
`gate_nb22.json`.


## 1. Imports and the release configuration

In [ ]:
import hashlib
import json
import random
import re
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from xml.etree import ElementTree

import numpy as np
import pandas as pd

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 3")

NB22_DIR = STAGE_D_DIR / "nb22_release"
NB22_DIR.mkdir(parents=True, exist_ok=True)

# Default mode certifies that the computational package is internally consistent while
# reporting unfinished camera-ready work separately. Set True only for the final
# submission run, after references, repeated seeds, dual coding, and manual checks exist.
ENFORCE_CAMERA_READY = False
# Add cloud-visible manuscript or bibliography paths here when they are outside
# PROJECT_ROOT/new_paper, old_paper, or manuscript. Files may be DOCX, BIB, BBL, TXT, or MD.
EXTRA_REFERENCE_SOURCES = []

# Preserve only the hand-written section of protocol_deviations.md. Remove every other
# managed output so a failed rerun cannot inherit a passing gate, checklist, or audit.
for name in ["artifact_manifest.csv", "restricted_data_hits.csv",
             "restricted_working_hits.csv", "release_allowlist.csv",
             "model_references.csv", "reference_audit.csv",
             "protocol_deviations.csv", "release_checklist.csv",
             "release_checklist.md", "environment.json",
             "run_config.json", "gate_nb22.json"]:
    stale = NB22_DIR / name
    if stale.is_file():
        stale.unlink()

# Anything matching these may NEVER be staged for publication.
RESTRICTED_PATTERNS = [
    (r"mimic", "MIMIC-CXR is under a PhysioNet data-use agreement"),
    (r"physionet", "PhysioNet credentialed data"),
    (r"\bphi\b|patient_name|mrn|accession", "possible protected health information"),
    (r"\.dcm$", "DICOM files can carry identifiers in their headers"),
    (r"faiss|\.index$", "a retrieval index built from restricted text is itself restricted"),
]
# File types that may be published.
PUBLISHABLE_SUFFIXES = {".csv", ".json", ".md", ".txt", ".tex", ".pdf", ".svg", ".png",
                        ".xlsx", ".ipynb", ".py", ".html", ".yaml", ".yml", ".lock"}
# Artifacts a reader needs to check the paper.
REQUIRED_ARTIFACTS = [
    ("nb17_statistics/all_metrics_with_ci.csv", "every metric with its interval"),
    ("nb19_external/paired_comparisons_final.csv", "every test after final Holm adjustment"),
    ("nb19_external/multiplicity_families_final.json", "final declared families F1-F5"),
    ("nb19_external/f5_external_comparisons.csv", "external confirmatory family F5"),
    ("nb19_external/decisive_comparison_final.json", "final F4-adjusted central verdict"),
    ("nb18_calibration/operating_points.csv", "fold-locked operating points"),
    ("nb19_external/external_metrics.csv", "external-cohort endpoint results"),
    ("nb19_external/e9c_threshold_transfer.csv", "internal-to-X2 threshold transfer"),
    ("nb20_interpretability/case_selection.csv", "locked qualitative sample"),
    ("nb20_interpretability/dual_coding_kappa.json", "human-coding status and agreement"),
    ("nb21_exports/tables/table10_operational_cost.csv", "operational-cost measurements"),
    ("nb21_exports/captions.md", "figure and table captions"),
    ("nb21_exports/acronym_table.csv", "Appendix A"),
    ("nb21_exports/traceability.csv", "the number-to-source map"),
    ("nb21_exports/figure6_case_panel_manifest.csv",
     "the fingerprinted qualitative-panel export"),
    ("nb21_exports/run_config.json", "NB 21 export provenance"),
    ("nb21_exports/gate_nb21.json", "NB 21 passing export gate"),
]
CLOSED_MODEL_PATTERN = re.compile(
    r"(?:^|[/_-])((?:gpt[-_.]?\d[0-9a-z._-]*)|"
    r"(?:o[1-9](?:[-_.][0-9a-z]+)*)|(?:claude[0-9a-z._-]*)|"
    r"(?:gemini[0-9a-z._-]*))(?:$|[/_-])",
    re.IGNORECASE)

MAX_HASH_BYTES = 200 * 1024 * 1024      # do not hash a multi-GB checkpoint to fill a column
# NB 21 deliberately permits a CSV-only supplement when openpyxl is unavailable. Require
# the workbook only when NB 21's own configuration says that it created one.
nb21_existing_config_path = STAGE_D_DIR / "nb21_exports" / "run_config.json"
if nb21_existing_config_path.is_file():
    try:
        nb21_existing_config = json.loads(
            nb21_existing_config_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        nb21_existing_config = {}
    if nb21_existing_config.get("workbook"):
        REQUIRED_ARTIFACTS.append(
            ("nb21_exports/supplementary_results.xlsx",
             "the numeric supplement declared by NB 21"))

e9c_path = STAGE_D_DIR / "nb19_external" / "e9c_threshold_transfer.csv"
if e9c_path.is_file():
    e9c_check = pd.read_csv(e9c_path)
    if ("threshold_source" in e9c_check and
            e9c_check["threshold_source"].astype(str).str.contains("HEADLINE",
                                                                    na=False).any()):
        REQUIRED_ARTIFACTS.append(
            ("nb21_exports/tables/table8b_x2_threshold_transfer.csv",
             "X2 performance at the internally transferred threshold"))
print("Release configuration loaded.")
print(f"  restricted patterns: {len(RESTRICTED_PATTERNS)}")
print(f"  required artifacts : {len(REQUIRED_ARTIFACTS)}")

## 2. Artifact manifest

Every file under the stage roots, with its size, a SHA-256 hash and the notebook directory that
owns it. Large binaries (model checkpoints) are listed with their size but not hashed — hashing
a 9 GB adapter to populate a spreadsheet column is a poor use of a login node, and the size plus
path is enough to identify it.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 5")

def sha256_of(path, limit=MAX_HASH_BYTES):
    size = path.stat().st_size
    if size > limit:
        return None, size
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest(), size


OWNER = {"stage_A": "Stage A (NB 00-04)", "stage_B": "Stage B (NB 05-12)",
         "stage_C": "Stage C (NB 13-16)", "stage_D": "Stage D (NB 17-22)"}

def build_manifest():
    rows = []
    manifest_path = NB22_DIR / "artifact_manifest.csv"
    for stage_dir in [STAGE_A_DIR, STAGE_B_DIR, STAGE_C_DIR, STAGE_D_DIR]:
        if not stage_dir.is_dir():
            continue
        stage_name = stage_dir.name
        for path in sorted(stage_dir.rglob("*")):
            if (not path.is_file() or path.name.startswith(".") or
                    path.resolve() == manifest_path.resolve()):
                continue
            digest, size = sha256_of(path)
            relative = path.relative_to(STAGE_ROOT)
            rows.append({
                "path": str(relative), "stage": stage_name,
                "owner": OWNER.get(stage_name, stage_name),
                "notebook_dir": relative.parts[1] if len(relative.parts) > 1 else "",
                "suffix": path.suffix.lower(), "bytes": size,
                "sha256": digest or "(not hashed: over the size limit)",
                "publishable_type": path.suffix.lower() in PUBLISHABLE_SUFFIXES})
    columns = ["path", "stage", "owner", "notebook_dir", "suffix", "bytes",
               "sha256", "publishable_type"]
    return pd.DataFrame(rows, columns=columns)


# Preliminary in-memory snapshot for the audits. The authoritative manifest is rebuilt and
# written only after every NB 22 output and the gate itself have been written.
manifest = build_manifest()
manifest_bytes = int(manifest["bytes"].sum()) if len(manifest) else 0
print(f"Preliminary manifest snapshot: {len(manifest):,} files, "
      f"{manifest_bytes / 1e9:.2f} GB total")
if len(manifest):
    print()
    print(manifest.groupby("stage").agg(files=("path", "count"),
                                        gigabytes=("bytes", lambda v: round(v.sum() / 1e9, 3))
                                        ).to_string())
    unhashed = int((manifest["sha256"].str.startswith("(not hashed")).sum())
    if unhashed:
        print(f"\n{unhashed} file(s) exceeded the {MAX_HASH_BYTES / 1e6:.0f} MB hashing limit "
              "and are listed by size and path only.")

## 3. Restricted-data check

The stage roots are controlled working storage, not a publication package. This section records
restricted-looking working paths for governance, then applies the blocking scan only to an
explicit release allowlist consisting of required results and NB 21 exports. A MIMIC derivative,
DICOM, PHI-like path, or restricted retrieval index on that allowlist blocks release.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 7")

def restricted_path_hits(rows):
    hits = []
    for row in rows:
        text = str(row["path"]).lower()
        for pattern, reason in RESTRICTED_PATTERNS:
            if re.search(pattern, text):
                hits.append({"path": row["path"], "pattern": pattern,
                             "reason": reason, "bytes": row["bytes"]})
                break
    return pd.DataFrame(hits, columns=["path", "pattern", "reason", "bytes"])


# The stage roots are controlled working storage, not a release bundle. Audit them so a
# restricted derivative is visible, but block only when it enters the explicit allowlist.
restricted_working = restricted_path_hits(manifest.to_dict("records"))
restricted_working.to_csv(NB22_DIR / "restricted_working_hits.csv", index=False)
required_paths = {f"stage_D/{name}" for name, _ in REQUIRED_ARTIFACTS}
release_mask = (manifest["path"].isin(required_paths)
                | manifest["path"].str.startswith("stage_D/nb21_exports/"))
release_candidates = manifest[release_mask].copy()
release_candidates[["path", "bytes", "sha256"]].to_csv(
    NB22_DIR / "release_allowlist.csv", index=False)
restricted = restricted_path_hits(release_candidates.to_dict("records"))
restricted.to_csv(NB22_DIR / "restricted_data_hits.csv", index=False)
if len(restricted):
    print(f"{len(restricted)} RELEASE-ALLOWLIST path(s) match a restricted-data pattern:")
    print(restricted.head(20).to_string(index=False))
    print("Remove them from the allowlist; never publish them.")
else:
    print(f"Release allowlist: {len(release_candidates)} files, no restricted path match.")
if len(restricted_working):
    print(f"Controlled working tree contains {len(restricted_working)} restricted-name "
          "artifact(s); they are inventoried but not release candidates.")

non_publishable = manifest[~manifest["publishable_type"]] if len(manifest) else pd.DataFrame()
if len(non_publishable):
    print()
    print(f"{len(non_publishable)} file(s) of a type not on the publishable list "
          f"({sorted(set(non_publishable['suffix']))[:8]}). These are working artifacts — "
          "checkpoints, caches, archives — and belong in the run directory, not the release.")

missing_required = [(name, why) for name, why in REQUIRED_ARTIFACTS
                    if not (STAGE_D_DIR / name).is_file()]
print()
if missing_required:
    print(f"{len(missing_required)} required artifact(s) missing:")
    for name, why in missing_required:
        print(f"  {name} — {why}")
else:
    print(f"All {len(REQUIRED_ARTIFACTS)} required artifacts are present.")

## 4. Reproducibility audit (R2.c)

Inspect structured model-ID fields in every run configuration, rather than searching narrative
notes for brand names. Closed-model identifiers block release. The Stage A registry must contain
an immutable commit hash for every model, every run configuration must record a seed, and the
software environment is saved for the manifest.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 9")

run_configs = {}
for path in sorted(STAGE_ROOT.rglob("run_config.json")):
    try:
        run_configs[str(path.relative_to(STAGE_ROOT))] = json.loads(
            path.read_text(encoding="utf-8"))
    except Exception as error:
        print(f"  could not read {path}: {type(error).__name__}")
print(f"Run configurations found: {len(run_configs)}")

closed_hits, unpinned, seedless, deterministic_seed_exempt, model_reference_rows = (
    [], [], [], [], [])
model_registry = {}
stage_a_paths_file = STAGE_A_DIR / "nb00_environment" / "stage_a_paths.json"
if stage_a_paths_file.is_file():
    model_registry = json.loads(stage_a_paths_file.read_text(encoding="utf-8")).get(
        "model_revisions", {})

def walk_fields(value, path=()):
    if isinstance(value, dict):
        for key, child in value.items():
            yield from walk_fields(child, path + (str(key),))
    elif isinstance(value, list):
        for index, child in enumerate(value):
            yield from walk_fields(child, path + (str(index),))
    else:
        yield path, value


def deterministic_without_seed(config):
    """True only when the saved contract proves inference is deterministic."""
    decoding = config.get("decoding") if isinstance(config, dict) else None
    return bool(isinstance(decoding, dict) and decoding.get("do_sample") is False
                and config.get("cache_fingerprint")
                and str(config.get("evaluation_mode") or "") in
                    {"out_of_fold", "inference", "evaluation"})


def has_seed_key(value):
    if isinstance(value, dict):
        return any((str(key).lower() == "seed" or str(key).lower().endswith("_seed"))
                   or has_seed_key(child)
                   for key, child in value.items())
    if isinstance(value, list):
        return any(has_seed_key(child) for child in value)
    return False


for name, config in run_configs.items():
    for field_path, value in walk_fields(config):
        if not field_path or not isinstance(value, str):
            continue
        key = field_path[-1].lower()
        is_model_field = (key == "model_id" or key.endswith("_model_id")
                          or key in {"base_model", "embedding_model"})
        if not is_model_field:
            continue
        model_reference_rows.append({"run_config": name,
                                     "field": ".".join(field_path),
                                     "model_id": value})
        match = CLOSED_MODEL_PATTERN.search(value)
        if match:
            closed_hits.append({"run_config": name,
                                "field": ".".join(field_path),
                                "model_id": value, "marker": match.group(1)})
    if not has_seed_key(config):
        if deterministic_without_seed(config):
            deterministic_seed_exempt.append(name)
        else:
            seedless.append(name)

model_references = pd.DataFrame(
    model_reference_rows, columns=["run_config", "field", "model_id"])
model_references.to_csv(NB22_DIR / "model_references.csv", index=False)

IMMUTABLE_REVISION = re.compile(r"^[0-9a-f]{40,64}$", re.IGNORECASE)
for model_id, revision in (model_registry or {}).items():
    if not revision or not IMMUTABLE_REVISION.fullmatch(str(revision).strip()):
        unpinned.append(model_id)

print()
if closed_hits:
    print(f"CLOSED-MODEL REFERENCES: {len(closed_hits)}")
    print(pd.DataFrame(closed_hits).to_string(index=False))
else:
    print("No closed-model identifier appears in any run configuration (R2.c).")

print(f"\nModel registry: {len(model_registry)} model(s)")
for model_id, revision in sorted((model_registry or {}).items()):
    marker = "pinned" if model_id not in unpinned else "NOT PINNED"
    print(f"  {model_id:<44} {str(revision)[:16]:<18} {marker}")
if unpinned:
    print(f"\n{len(unpinned)} model(s) without a pinned revision. A model id alone is not "
          "reproducible: the weights behind it can be replaced.")
if deterministic_seed_exempt:
    print(f"\n{len(deterministic_seed_exempt)} deterministic, fingerprinted "
          f"inference configuration(s) need no random seed: "
          f"{deterministic_seed_exempt[:5]}")
if seedless:
    print(f"\n{len(seedless)} stochastic or undeclared run configuration(s) record no "
          f"seed: {seedless[:5]}")

environment = {
    "python": sys.version.split()[0],
    "platform": sys.platform,
    "numpy": np.__version__, "pandas": pd.__version__,
    "written_utc": datetime.now(timezone.utc).isoformat(),
}
for module_name in ["torch", "transformers", "peft", "sklearn", "scipy", "matplotlib", "PIL"]:
    try:
        module = __import__(module_name)
        environment[module_name] = getattr(module, "__version__", "unknown")
    except ImportError:
        environment[module_name] = "not installed in this kernel"
try:
    environment["git_commit"] = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=str(PROJECT_ROOT), capture_output=True,
        text=True, timeout=10).stdout.strip() or "not a git repository"
except Exception:
    environment["git_commit"] = "unavailable"
sd.write_json_atomic(NB22_DIR / "environment.json", environment)
print()
print(json.dumps(environment, indent=2))

## 5. Reference audit (R1.6)

Audit bibliography and manuscript sources, including Word documents. Preprints without a DOI are
marked `replace`; entries that still read as preprints despite a DOI are marked `verify` and also
block release until resolved. Model cards may be cited as software only when their citation or the
pinned Stage A registry supplies an immutable revision.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 11")

# Model cards that have no peer-reviewed version and are cited as software, with hashes.
SOFTWARE_ARTIFACTS = ["nv-reason-cxr", "nvidia/nv-reason", "cxformer",
                      "m42-health/cxformer", "medgemma", "google/medgemma",
                      "biomedclip", "microsoft/biomedclip", "torchxrayvision"]
ARXIV = re.compile(r"arxiv[.: ]?\s*(\d{4}\.\d{4,5})", re.IGNORECASE)
PREPRINT_MARKERS = ["preprint", "arxiv", "biorxiv", "medrxiv", "under review",
                    "submitted to", "openreview"]
VERSION_HASH = re.compile(
    r"(?:revision|commit|sha(?:256)?|version)[^0-9a-f]{0,24}([0-9a-f]{7,64})(?![0-9a-f])",
    re.IGNORECASE)

search_roots = [PROJECT_ROOT / "new_paper", PROJECT_ROOT / "old_paper",
                PROJECT_ROOT / "manuscript"]
bibliography_files = []
for root in search_roots:
    if root.is_dir():
        for pattern in ["*.bib", "*.bbl", "references*.txt", "references*.md",
                        "*MANUSCRIPT*.txt", "*manuscript*.md", "*.docx"]:
            bibliography_files.extend(sorted(root.rglob(pattern)))
# Also accept explicit bibliography/manuscript files placed at the project root, without
# recursively scanning notebooks and protocol notes as if they were citations.
for pattern in ["*.bib", "*.bbl", "references*.txt", "references*.md",
                "*manuscript*.docx", "*MANUSCRIPT*.docx", "submit*.docx",
                "revision*.docx"]:
    bibliography_files.extend(sorted(PROJECT_ROOT.glob(pattern)))
for source in EXTRA_REFERENCE_SOURCES:
    candidate = Path(source).expanduser()
    if candidate.is_file():
        bibliography_files.append(candidate)
    else:
        print(f"  explicit reference source is unavailable: {candidate}")
bibliography_files = list(dict.fromkeys(bibliography_files))
print(f"Bibliography / manuscript sources found: {[p.name for p in bibliography_files]}")

def document_paragraphs(path):
    if path.suffix.lower() != ".docx":
        return path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        with zipfile.ZipFile(path) as archive:
            root = ElementTree.fromstring(archive.read("word/document.xml"))
    except (OSError, KeyError, zipfile.BadZipFile, ElementTree.ParseError) as exc:
        raise RuntimeError(f"Could not read Word manuscript {path}") from exc
    paragraphs = []
    for paragraph in root.iter("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}p"):
        words = [node.text or "" for node in paragraph.iter(
            "{http://schemas.openxmlformats.org/wordprocessingml/2006/main}t")]
        text = "".join(words).strip()
        if text:
            paragraphs.append(text)
    return paragraphs


def registry_hash_for(lowered_chunk):
    matches = []
    for model_id, revision in (model_registry or {}).items():
        model_text = str(model_id).lower()
        tokens = [model_text, model_text.split("/")[-1]]
        tokens.extend(marker for marker in SOFTWARE_ARTIFACTS if marker in model_text)
        if any(token in lowered_chunk for token in tokens) and revision and \
                IMMUTABLE_REVISION.fullmatch(str(revision).strip()):
            matches.append((model_id, str(revision)))
    return matches


audit_rows = []
for path in bibliography_files:
    paragraphs = document_paragraphs(path)
    text = "\n".join(paragraphs)
    # Parse one bibliographic entry at a time so a hash in one software citation cannot
    # accidentally satisfy another entry in the same file.
    if path.suffix == ".bib":
        chunks = [chunk for chunk in re.split(r"(?=^@)", text, flags=re.MULTILINE)
                  if chunk.strip()]
    elif path.suffix == ".bbl":
        chunks = [chunk for chunk in re.split(r"(?=\\bibitem)", text) if chunk.strip()]
    elif path.suffix.lower() == ".docx":
        headings = [i for i, paragraph in enumerate(paragraphs)
                    if paragraph.strip().lower() in {"references", "bibliography"}]
        reference_paragraphs = paragraphs[headings[-1] + 1:] if headings else paragraphs
        chunks = [paragraph for paragraph in reference_paragraphs if paragraph.strip()]
    else:
        chunks = [chunk for chunk in re.split(r"\n\s*\n", text) if chunk.strip()]
    for index, chunk in enumerate(chunks):
        lowered = chunk.lower()
        if not any(marker in lowered for marker in
                   PREPRINT_MARKERS + SOFTWARE_ARTIFACTS +
                   ["doi", "@article", "@inproceedings"]):
            continue
        arxiv_id = ARXIV.search(chunk)
        is_software = any(marker in lowered for marker in SOFTWARE_ARTIFACTS)
        has_doi = bool(re.search(r"\bdoi\b|10\.\d{4,9}/", lowered))
        is_preprint = bool(arxiv_id) or any(marker in lowered
                                            for marker in PREPRINT_MARKERS)
        registry_hashes = registry_hash_for(lowered) if is_software else []
        has_version_hash = bool(VERSION_HASH.search(chunk) or registry_hashes)
        if is_software:
            verdict = "software artifact" if has_version_hash else "software hash missing"
            action = ("cite as software with a pinned immutable revision hash, not as "
                      "evidence for a clinical claim")
        elif is_preprint and not has_doi:
            verdict, action = "replace", ("find the journal or conference version, or remove "
                                          "the citation")
        elif is_preprint and has_doi:
            verdict, action = "verify", ("has a DOI but reads as a preprint; confirm which "
                                         "version is being cited")
        else:
            verdict, action = "ok", ""
        audit_rows.append({
            "source_file": (str(path.relative_to(PROJECT_ROOT))
                            if path.is_relative_to(PROJECT_ROOT) else str(path)),
            "chunk": index,
            "excerpt": " ".join(chunk.split())[:180],
            "arxiv_id": arxiv_id.group(1) if arxiv_id else "",
            "has_doi": has_doi, "has_version_hash": has_version_hash,
            "hash_source": ("citation text" if VERSION_HASH.search(chunk) else
                            ";".join(f"{model}@{revision}"
                                     for model, revision in registry_hashes)),
            "verdict": verdict, "action": action})

reference_columns = ["source_file", "chunk", "excerpt", "arxiv_id", "has_doi",
                     "has_version_hash", "hash_source", "verdict", "action"]
reference_audit = pd.DataFrame(audit_rows, columns=reference_columns)
reference_audit.to_csv(NB22_DIR / "reference_audit.csv", index=False)
if len(reference_audit):
    print()
    print(reference_audit["verdict"].value_counts().to_string())
    to_replace = reference_audit[reference_audit["verdict"] == "replace"]
    if len(to_replace):
        print()
        print(f"{len(to_replace)} citation(s) to replace (R1.6):")
        for row in to_replace.head(10).to_dict("records"):
            print(f"  [{row['source_file']}] {row['excerpt'][:110]}")
    software = reference_audit[reference_audit["verdict"].isin(
        ["software artifact", "software hash missing"])]
    if len(software):
        print()
        print(f"{len(software)} declared software-artifact citation(s) were audited as "
              "software rather than clinical evidence.")
        print("  They must be cited for what the software IS, with a version hash; the")
        print("  manuscript should say so explicitly rather than leaving them looking like")
        print("  overlooked preprints.")
else:
    print()
    print("NO bibliography or manuscript source was found, so the reference audit could not "
          "run.")
    print("  An empty audit and a passing audit look identical in a checklist. This one is")
    print("  empty: R1.6 remains unverified until a bibliography exists to check.")

## 6. Protocol-deviation log

Protocol §8.9 freezes sections 6–8 before the experiments run, and requires every later change
to be recorded with a date and a reason.

The log is assembled from what the pipeline actually reports: gate warnings that mark an arm as
unavailable, absent cohorts, declared-but-not-executed arms. Those *are* deviations, and
collecting them automatically is what stops the log from being written retrospectively by
whoever remembers least.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 13")

deviations = []


def note(area, deviation, reason, impact):
    deviations.append({"date": datetime.now(timezone.utc).date().isoformat(), "area": area,
                       "deviation": deviation, "reason": reason, "impact": impact})


# ---- Harvest from the gates every stage wrote ------------------------------------------
gate_files = [path for path in sorted(STAGE_ROOT.rglob("gate_nb*.json"))
              if path.name != "gate_nb22.json"]
gate_summary = []
for path in gate_files:
    try:
        gate = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    gate_summary.append({"gate": path.stem,
                         "path": str(path.relative_to(STAGE_ROOT)),
                         "passed": gate.get("passed"),
                         "n_failures": len(gate.get("failures") or []),
                         "n_warnings": len(gate.get("warnings") or [])})
    for warning in (gate.get("warnings") or []):
        lowered = str(warning).lower()
        if any(marker in lowered for marker in
               ["absent", "not run", "not executed", "missing", "could not", "not yet",
                "never prepared", "declared but"]):
            note(path.stem, str(warning)[:300], "recorded automatically from the gate warning",
                 "reduces the scope of a protocol arm")

gates = pd.DataFrame(gate_summary)
if len(gates):
    print(gates.to_string(index=False))
    failed = gates[gates["passed"].ne(True)]
    if len(failed):
        print()
        print(f"{len(failed)} gate(s) are currently FAILING: {failed['gate'].tolist()}")
        print("  A release cannot be assembled while a gate fails; fix the notebook rather")
        print("  than recording the failure as a deviation.")

# ---- Standing deviations known from the design ------------------------------------------
if unpinned:
    note("reproducibility", f"model revisions not pinned: {unpinned}",
         "the registry recorded no revision hash",
         "the exact weights cannot be recovered by a third party (R2.c)")
if not bibliography_files:
    note("references", "reference audit could not run",
         "no bibliography or manuscript source was found",
         "R1.6 remains unverified")
if missing_required:
    note("artifacts", f"missing required artifacts: {[n for n, _ in missing_required]}",
         "the producing notebook has not run or did not finish",
         "a reviewer cannot trace every number in the manuscript")

deviation_frame = pd.DataFrame(
    deviations, columns=["date", "area", "deviation", "reason", "impact"])
deviation_path = NB22_DIR / "protocol_deviations.md"
manual_body = ""
if deviation_path.is_file():
    previous = deviation_path.read_text(encoding="utf-8")
    marker = "## Added by hand"
    if marker in previous:
        manual_body = previous.split(marker, 1)[1].strip()
lines = ["# Protocol deviation log", "",
         "Protocol section 8.9 freezes sections 6-8 before the experiments run and requires "
         "every later change to be recorded with a date and a reason. This log is assembled "
         "automatically from the pipeline's own gates, then extended by hand.", ""]
if len(deviation_frame):
    lines += ["| date | area | deviation | reason | impact |",
              "| --- | --- | --- | --- | --- |"]
    def markdown_cell(value):
        return str(value).replace("|", "\\|").replace("\n", " ")
    for row in deviations:
        lines.append("| " + " | ".join(markdown_cell(row[column]) for column in
                    ["date", "area", "deviation", "reason", "impact"]) + " |")
else:
    lines.append("No deviations were detected automatically. Add any made by hand below.")
lines += ["", "## Added by hand", ""]
if manual_body:
    lines.append(manual_body)
else:
    lines += ["_Record here anything the automatic pass cannot see: a changed threshold, a "
              "re-run with different data, or a decision taken after seeing a result._"]
lines.append("")
deviation_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
deviation_frame.to_csv(NB22_DIR / "protocol_deviations.csv", index=False)
print()
print(f"protocol_deviations.md: {len(deviations)} automatically detected deviation(s)")
for row in deviations[:6]:
    print(f"  [{row['area']}] {row['deviation'][:110]}")

## 7. Release checklist

A single markdown file that says, item by item, what is ready and what is not — generated from
the checks above so it cannot claim readiness the pipeline does not support.

The language-pass items (R1.10) are the only ones a notebook cannot verify. They are listed as
manual with an explicit owner, rather than silently marked done.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 15")

def status(condition, ready="READY", blocked="BLOCKED"):
    return ready if condition else blocked


def read_json(path):
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return {}


# One exact gate per notebook. Duplicate matches are treated as ambiguous provenance, not
# resolved by whichever path happens to sort first.
STAGE_D_OUTPUTS = {
    17: STAGE_D_DIR / "nb17_statistics",
    18: STAGE_D_DIR / "nb18_calibration",
    19: STAGE_D_DIR / "nb19_external",
    20: STAGE_D_DIR / "nb20_interpretability",
    21: STAGE_D_DIR / "nb21_exports",
}
stage_d_gates, identity_failures = {}, []
for number, directory in STAGE_D_OUTPUTS.items():
    candidates = [directory / f"gate_nb{number:02d}.json",
                  directory / f"gate_nb{number}.json"]
    matches = list(dict.fromkeys(path for path in candidates if path.is_file()))
    if len(matches) != 1:
        identity_failures.append(
            f"NB {number}: expected exactly one gate in {directory}, found {len(matches)}.")
        stage_d_gates[number] = {"path": None, "payload": {}}
        continue
    payload = read_json(matches[0])
    stage_d_gates[number] = {"path": str(matches[0]), "payload": payload}
    if payload.get("passed") is not True:
        identity_failures.append(f"NB {number}: gate is missing, unreadable, or not passing.")


def gate_passed(number):
    return stage_d_gates.get(int(number), {}).get("payload", {}).get("passed") is True


nb17_config = read_json(STAGE_D_OUTPUTS[17] / "run_config.json")
nb18_config = read_json(STAGE_D_OUTPUTS[18] / "run_config.json")
nb19_config = read_json(STAGE_D_OUTPUTS[19] / "run_config.json")
nb20_config = read_json(STAGE_D_OUTPUTS[20] / "run_config.json")
nb21_config = read_json(STAGE_D_OUTPUTS[21] / "run_config.json")
multiplicity_final = read_json(
    STAGE_D_OUTPUTS[19] / "multiplicity_families_final.json")


def require_identity(label, expected, observed):
    expected_text = str(expected or "").strip()
    observed_text = str(observed or "").strip()
    if not expected_text:
        identity_failures.append(f"{label}: upstream identity value is empty.")
    elif observed_text != expected_text:
        identity_failures.append(
            f"{label}: expected {expected_text!r}, observed {observed_text!r}.")


reference_arm = nb17_config.get("reference_arm")
bootstrap_fingerprint = nb17_config.get("bootstrap_fingerprint")
selection_hash = nb20_config.get("selection_hash")
case_table_fingerprint = nb20_config.get("case_table_fingerprint")
family_counts = multiplicity_final.get("family_counts") or {}

require_identity("NB 18 run config bootstrap", bootstrap_fingerprint,
                 nb18_config.get("bootstrap_fingerprint"))
require_identity("NB 18 gate bootstrap", bootstrap_fingerprint,
                 stage_d_gates[18]["payload"].get("bootstrap_fingerprint"))
require_identity("NB 19 run config bootstrap", bootstrap_fingerprint,
                 nb19_config.get("nb18_bootstrap_fingerprint"))
require_identity("NB 19 gate bootstrap", bootstrap_fingerprint,
                 stage_d_gates[19]["payload"].get("nb18_bootstrap_fingerprint"))
require_identity("NB 20 reference arm", reference_arm, nb20_config.get("reference_arm"))
for label, payload in [("NB 21 run config", nb21_config),
                       ("NB 21 gate", stage_d_gates[21]["payload"])]:
    require_identity(f"{label} reference arm", reference_arm, payload.get("reference_arm"))
    require_identity(f"{label} bootstrap", bootstrap_fingerprint,
                     payload.get("bootstrap_fingerprint"))
    require_identity(f"{label} selection hash", selection_hash,
                     payload.get("nb20_selection_hash"))
    require_identity(f"{label} case-table fingerprint", case_table_fingerprint,
                     payload.get("nb20_case_table_fingerprint"))
    if payload.get("final_family_counts") != family_counts:
        identity_failures.append(
            f"{label}: final multiplicity-family counts disagree with NB 19.")


# Intervals are required only where an endpoint estimate exists. Endpoint-unusable arms
# legitimately retain NaN estimates and must not make the release gate claim divergence.
metrics_path = STAGE_D_OUTPUTS[17] / "all_metrics_with_ci.csv"
metrics_ready, metric_issues = False, []
if metrics_path.is_file():
    metric_check = pd.read_csv(metrics_path)
    if not len(metric_check) or "arm" not in metric_check:
        metric_issues.append("empty artifact or missing arm column")
    for estimate, low, high in [
            ("mrale_mae", "mrale_mae_ci_low", "mrale_mae_ci_high"),
            ("covid_auroc", "covid_auroc_ci_low", "covid_auroc_ci_high")]:
        missing = [column for column in [estimate, low, high]
                   if column not in metric_check]
        if missing:
            metric_issues.append(f"{estimate}: missing {missing}")
            continue
        finite_estimate = np.isfinite(pd.to_numeric(metric_check[estimate], errors="coerce"))
        finite_interval = (np.isfinite(pd.to_numeric(metric_check[low], errors="coerce")) &
                           np.isfinite(pd.to_numeric(metric_check[high], errors="coerce")))
        if finite_estimate.any() and not finite_interval[finite_estimate].all():
            metric_issues.append(f"{estimate}: a finite estimate lacks its interval")
    metrics_ready = len(metric_check) > 0 and not metric_issues


final_paired_path = STAGE_D_OUTPUTS[19] / "paired_comparisons_final.csv"
paired_ready, paired_issues = False, []
if final_paired_path.is_file():
    paired_check = pd.read_csv(final_paired_path)
    required = ["test", "paired_unit", "family", "p_raw", "p_adjusted",
                "p_reportable"]
    missing = [column for column in required if column not in paired_check]
    if not len(paired_check):
        paired_issues.append("artifact is empty")
    elif missing:
        paired_issues.append(f"missing columns {missing}")
    else:
        finite = paired_check[pd.to_numeric(
            paired_check["p_raw"], errors="coerce").notna()].copy()
        if not len(finite):
            paired_issues.append("no finite p-values")
        metadata = ["test", "paired_unit", "family"]
        if len(finite) and finite[metadata].isna().any().any():
            paired_issues.append("finite p-value without complete metadata")
        confirmatory = finite[finite["family"].astype(str) != "EXPLORATORY"]
        adjusted = pd.to_numeric(confirmatory["p_adjusted"], errors="coerce")
        reportable = pd.to_numeric(confirmatory["p_reportable"], errors="coerce")
        if len(confirmatory) and (adjusted.isna().any() or reportable.isna().any() or
                                 not np.allclose(adjusted, reportable, rtol=0, atol=1e-12)):
            paired_issues.append("confirmatory reportable p-values are not final Holm values")
        exploratory = finite[finite["family"].astype(str) == "EXPLORATORY"]
        raw = pd.to_numeric(exploratory["p_raw"], errors="coerce")
        shown = pd.to_numeric(exploratory["p_reportable"], errors="coerce")
        if len(exploratory) and (shown.isna().any() or
                                not np.allclose(raw, shown, rtol=0, atol=1e-12)):
            paired_issues.append("exploratory reportable p-values differ from raw values")
        paired_ready = not paired_issues


nb17_release_config = nb17_config
seed_stability_ready = (
    nb17_release_config.get("repeated_seed_complete") is True and
    (STAGE_D_OUTPUTS[17] / "seed_stability.csv").is_file())

dual_info = read_json(STAGE_D_OUTPUTS[20] / "dual_coding_kappa.json")
case_selection_path = STAGE_D_OUTPUTS[20] / "case_selection.csv"
panel_manifest_path = STAGE_D_OUTPUTS[21] / "figure6_case_panel_manifest.csv"
case_selection = (pd.read_csv(case_selection_path) if case_selection_path.is_file()
                  else pd.DataFrame())
panel_manifest = (pd.read_csv(panel_manifest_path) if panel_manifest_path.is_file()
                  else pd.DataFrame())
panels_ready = bool(len(case_selection) and len(panel_manifest) == len(case_selection))
if panels_ready:
    panels_ready = ("selection_hash" in panel_manifest and
                    set(panel_manifest["selection_hash"].dropna().astype(str)) ==
                    {str(selection_hash)} and
                    "exported" in panel_manifest and
                    panel_manifest["exported"].astype(str).str.lower().eq("true").all())

trace_path = STAGE_D_OUTPUTS[21] / "traceability.csv"
traceability = pd.read_csv(trace_path) if trace_path.is_file() else pd.DataFrame()
traceability_ready = bool(len(traceability) and "traced" in traceability and
                          traceability["traced"].astype(str).str.lower()
                          .isin(["true", "1"]).all())

figure_dir = STAGE_D_OUTPUTS[21] / "figures"
expected_vectors = [str(name) for name in nb21_config.get("vector_figures") or []]
figures_ready = bool(expected_vectors and all((figure_dir / name).is_file()
                                              for name in expected_vectors) and
                     any(Path(name).suffix.lower() == ".pdf" for name in expected_vectors) and
                     any(Path(name).suffix.lower() == ".svg" for name in expected_vectors))

acronym_path = STAGE_D_OUTPUTS[21] / "acronym_table.csv"
acronyms = pd.read_csv(acronym_path) if acronym_path.is_file() else pd.DataFrame()
acronyms_ready = bool(len(acronyms) and "Defined before use" in acronyms and
                      acronyms["Defined before use"].fillna("").astype(str)
                      .str.lower().eq("yes").all() and
                      not (nb21_config.get("acronym_ordering_problems") or []))

external_path = STAGE_D_OUTPUTS[19] / "external_metrics.csv"
external_check = pd.read_csv(external_path) if external_path.is_file() else pd.DataFrame()
external_ready = bool(len(external_check) and {"cohort", "arm", "n"} <=
                      set(external_check.columns))

reference_blocking = (reference_audit["verdict"].isin(
    ["replace", "verify"]).sum() if len(reference_audit) else 0)
software_hash_missing = (reference_audit["verdict"].eq(
    "software hash missing").sum() if len(reference_audit) else 0)

checklist_items = [
    ("Statistics", "finite endpoint estimates have patient-level intervals",
     status(metrics_ready and gate_passed(17),
            blocked="; ".join(metric_issues) or "NB 17 NOT PASSING"), "NB 17"),
    ("Statistics", "every finite p-value carries metadata and the final reportable value",
     status(paired_ready and gate_passed(19),
            blocked="; ".join(paired_issues) or "NB 19 NOT PASSING"), "NB 19"),
    ("Statistics", "multiplicity families declared and Holm applied within each (8.6)",
     status(bool(family_counts) and gate_passed(19)), "NB 19"),
    ("Stability", "headline arms completed registered repeated seeds 7, 42, and 1234",
     status(seed_stability_ready), "upstream training notebooks / NB 17"),
    ("Calibration", "ROC/PR curves, reliability diagrams and operating points",
     status((STAGE_D_OUTPUTS[18] / "operating_points.csv").is_file() and
            gate_passed(18)), "NB 18"),
    ("External", "external validation evaluated on at least one cohort (R1.5)",
     status(external_ready and gate_passed(19)), "NB 19"),
    ("Interpretability", "every locked case panel entered the fingerprinted export",
     status(panels_ready and gate_passed(20) and gate_passed(21)), "NB 20 / NB 21"),
    ("Interpretability", "failure taxonomy dual-coded with Cohen's kappa (E10b)",
     status(dual_info.get("available") is True and
            dual_info.get("taxonomy_final_available") is True,
            blocked="NEEDS COMPLETE DUAL CODING AND ADJUDICATION"), "two authors"),
    ("Figures", "all declared vector figures exist as PDF and SVG (R1.9)",
     status(figures_ready and gate_passed(21)), "NB 21"),
    ("Tables", "every numeric cell traces to an identity-matched locked artifact",
     status(traceability_ready and gate_passed(21)), "NB 21"),
    ("Appendix", "acronym table passed the manuscript first-use check (R1.8)",
     status(acronyms_ready), "NB 21 / authors"),
    ("Reproducibility", "all structured model identifiers are open-weight/local",
     status(not closed_hits), "NB 00 / design"),
    ("Reproducibility", "every model revision is pinned to an immutable hash (R2.c)",
     status(bool(model_registry) and not unpinned,
            blocked=("NO MODEL REGISTRY" if not model_registry
                     else f"{len(unpinned)} UNPINNED")), "NB 00"),
    ("Reproducibility", "environment recorded (versions, git commit)",
     status((NB22_DIR / "environment.json").is_file()), "NB 22"),
    ("Data governance", "release allowlist contains no restricted-data path",
     status(len(restricted) == 0, blocked=f"{len(restricted)} PATH(S) MATCH"), "NB 22"),
    ("References", "no preprint is unresolved or cited as evidence (R1.6)",
     status(len(reference_audit) > 0 and reference_blocking == 0,
            blocked=("NOT AUDITED (no bibliography found)" if not len(reference_audit)
                     else f"{reference_blocking} REPLACE/VERIFY")), "authors"),
    ("References", "software artifacts resolve to immutable version hashes",
     status(len(reference_audit) > 0 and software_hash_missing == 0,
            blocked="NOT AUDITED OR HASH MISSING"), "authors"),
    ("Protocol", "deviation log present and current (8.9)",
     status((NB22_DIR / "protocol_deviations.md").is_file()), "NB 22"),
    ("Protocol", "NB 17-21 gates and cross-notebook fingerprints agree",
     status(not identity_failures,
            blocked=f"{len(identity_failures)} IDENTITY/GATE ISSUE(S)"), "NB 17-21"),
    ("Language", "full proofread pass (R1.10)", "MANUAL", "authors"),
    ("Language", "consistent terminology: mRALE, PCR-confirmed, out-of-fold", "MANUAL",
     "authors"),
    ("Language", "every claim in the abstract is supported by a traced table row", "MANUAL",
     "authors"),
]

checklist = pd.DataFrame(checklist_items, columns=["Area", "Item", "Status", "Owner"])
checklist.to_csv(NB22_DIR / "release_checklist.csv", index=False)

blocked = checklist[~checklist["Status"].isin(["READY", "MANUAL"])]
manual = checklist[checklist["Status"] == "MANUAL"]
lines = ["# Release checklist", "",
         f"Generated {datetime.now(timezone.utc).isoformat()} from locked pipeline artifacts. "
         "READY means both the producing gate and its identity contract passed.", "",
         f"**{len(checklist) - len(blocked) - len(manual)} ready, {len(blocked)} blocked, "
         f"{len(manual)} manual.**", "",
         "| area | item | status | owner |", "| --- | --- | --- | --- |"]
for row in checklist_items:
    safe = [str(value).replace("|", "\\|").replace("\n", " ") for value in row]
    lines.append("| " + " | ".join(safe) + " |")
if len(blocked):
    lines += ["", "## Blocking items", ""]
    for row in blocked.to_dict("records"):
        lines.append(f"- **{row['Item']}** — {row['Status']} (owner: {row['Owner']})")
lines += ["", "## What the pipeline cannot check", "",
          "The language items are genuinely manual. They are listed rather than assumed so "
          "that nobody mistakes a green automatic checklist for a proofread manuscript.", ""]
(NB22_DIR / "release_checklist.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

print(checklist.to_string(index=False))
if identity_failures:
    print("\nCross-notebook identity issues:")
    for issue in identity_failures:
        print("  -", issue)
print()
print(f"{len(checklist) - len(blocked) - len(manual)} ready, {len(blocked)} blocked, "
      f"{len(manual)} manual")


## 8. Run configuration and gate

NB 22 has two explicit states. The default **analysis gate** certifies computational integrity:
core artifacts, upstream gates and fingerprints, model provenance, and the restricted-data
allowlist. The separate **camera-ready readiness** state records unfinished repeated-seed runs,
reference verification, dual coding, acronym checking, and author sign-offs without making an
otherwise valid analysis notebook crash. Before submission, set `ENFORCE_CAMERA_READY = True`;
then those release blockers become assertion failures. The manifest is rebuilt last so it hashes
the final configuration and gate.


In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 22 before code cell 17")

hard_failures, release_blockers, warnings = [], [], []

if len(restricted):
    hard_failures.append(
        f"{len(restricted)} release-allowlist path(s) match a restricted-data pattern, e.g. "
        f"{restricted.iloc[0]['path']} ({restricted.iloc[0]['reason']}). Remove them from "
        "the publication bundle; controlled working artifacts are audited separately.")
else:
    print(f"RESTRICTED-DATA CHECK PASSED: {len(release_candidates):,} release candidates, "
          "no match.")

if closed_hits:
    hard_failures.append(
        f"{len(closed_hits)} structured model-id field(s) reference a closed model "
        f"({sorted({hit['marker'] for hit in closed_hits})}). Every arm in this revision "
        "must be open-weight and locally hosted.")
else:
    print("REPRODUCIBILITY CHECK PASSED: no closed model in a structured model-id field.")

if not model_registry:
    hard_failures.append(
        "The model registry is empty, so immutable revision pinning is unverified.")
elif unpinned:
    hard_failures.append(
        f"{len(unpinned)} model(s) lack an immutable commit revision: {unpinned}.")
if seedless:
    hard_failures.append(
        f"{len(seedless)} stochastic or undeclared run configuration(s) record no random "
        f"seed: {seedless[:8]}.")
if deterministic_seed_exempt:
    warnings.append(
        f"{len(deterministic_seed_exempt)} deterministic, fingerprinted inference "
        "configuration(s) omit a random seed by design: "
        f"{deterministic_seed_exempt[:8]}.")
if missing_required:
    hard_failures.append(
        f"{len(missing_required)} core artifact(s) are missing: "
        f"{[name for name, _ in missing_required]}.")
if identity_failures:
    hard_failures.append(
        f"{len(identity_failures)} NB 17-21 gate/fingerprint issue(s): "
        + "; ".join(identity_failures[:5]))

if not len(reference_audit):
    release_blockers.append(
        "The reference audit found no bibliography/manuscript source; R1.6 is unverified. "
        "Copy the manuscript to a searched project folder or add its cloud path to "
        "EXTRA_REFERENCE_SOURCES.")
else:
    unresolved = reference_audit[reference_audit["verdict"].isin(["replace", "verify"])]
    if len(unresolved):
        release_blockers.append(
            f"{len(unresolved)} citation(s) still require replacement or manual version "
            "verification (R1.6).")
    missing_hash = reference_audit[
        reference_audit["verdict"] == "software hash missing"]
    if len(missing_hash):
        release_blockers.append(
            f"{len(missing_hash)} software citation(s) lack an immutable version/commit hash.")

if len(non_publishable):
    warnings.append(
        f"{len(non_publishable)} controlled working artifact(s) have a non-publishable "
        "type. They are outside the release allowlist.")
if len(restricted_working):
    warnings.append(
        f"{len(restricted_working)} controlled working path(s) match a restricted-data "
        "pattern. They are inventoried in restricted_working_hits.csv and excluded from "
        "the release allowlist.")

blocked_items = checklist[~checklist["Status"].isin(["READY", "MANUAL"])]
release_only = (
    blocked_items["Area"].isin(["Stability", "Appendix", "References"]) |
    blocked_items["Item"].str.contains("dual-coded", case=False, na=False))
hard_checklist_items = blocked_items[~release_only]
release_checklist_items = blocked_items[release_only]
if len(hard_checklist_items):
    hard_failures.append(
        f"{len(hard_checklist_items)} computational checklist item(s) are blocked: "
        + "; ".join(f"{row['Item']} ({row['Status']})"
                    for row in hard_checklist_items.to_dict("records")[:5]))
if len(release_checklist_items):
    release_blockers.append(
        f"{len(release_checklist_items)} camera-ready checklist item(s) are blocked: "
        + "; ".join(f"{row['Item']} ({row['Status']})"
                    for row in release_checklist_items.to_dict("records")[:5]))

manual_items = checklist[checklist["Status"] == "MANUAL"]
warnings.append(
    f"{len(manual_items)} checklist item(s) require author sign-off. A passing analysis "
    "gate is not a proofread manuscript.")

analysis_passed = not hard_failures
automated_release_ready = analysis_passed and not release_blockers
blocking_failures = list(hard_failures)
if ENFORCE_CAMERA_READY:
    blocking_failures.extend(release_blockers)

run_payload = sd.provenance_stamp(
    "22_reproducibility_and_reference_audit.ipynb",
    {"gate_mode": "camera_ready" if ENFORCE_CAMERA_READY else "analysis",
     "analysis_passed": analysis_passed,
     "automated_release_ready": automated_release_ready,
     "n_artifacts_preliminary": int(len(manifest)),
     "preliminary_total_bytes": int(manifest["bytes"].sum()) if len(manifest) else 0,
     "n_release_candidates": int(len(release_candidates)),
     "n_restricted_release_hits": int(len(restricted)),
     "n_restricted_working_hits": int(len(restricted_working)),
     "n_structured_model_references": int(len(model_references)),
     "n_closed_model_hits": len(closed_hits),
     "unpinned_models": unpinned,
     "seedless_configs": seedless,
     "deterministic_seed_exempt_configs": deterministic_seed_exempt,
     "missing_required_artifacts": [name for name, _ in missing_required],
     "n_references_audited": int(len(reference_audit)),
     "reference_verdicts": (reference_audit["verdict"].value_counts().to_dict()
                            if len(reference_audit) else {}),
     "n_deviations": len(deviations),
     "gates": gate_summary,
     "stage_d_identity": {
         "reference_arm": reference_arm,
         "bootstrap_fingerprint": bootstrap_fingerprint,
         "nb20_selection_hash": selection_hash,
         "nb20_case_table_fingerprint": case_table_fingerprint,
         "final_family_counts": family_counts},
     "identity_failures": identity_failures,
     "release_blockers": release_blockers,
     "checklist_ready": int((checklist["Status"] == "READY").sum()),
     "checklist_blocked": int(len(blocked_items)),
     "checklist_manual": int(len(manual_items)),
     "environment": environment})
sd.write_json_atomic(NB22_DIR / "run_config.json", run_payload)


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("COMPUTATIONAL FAILURES", hard_failures)
print()
report("CAMERA-READY BLOCKERS", release_blockers)

gate_payload = {
    "passed": not blocking_failures,
    "gate_mode": "camera_ready" if ENFORCE_CAMERA_READY else "analysis",
    "analysis_passed": analysis_passed,
    "automated_release_ready": automated_release_ready,
    "failures": blocking_failures,
    "computational_failures": hard_failures,
    "release_blockers": release_blockers,
    "warnings": warnings,
    "manual_items_pending": int(len(manual_items)),
    "reference_arm": reference_arm,
    "bootstrap_fingerprint": bootstrap_fingerprint,
    "nb20_selection_hash": selection_hash,
    "nb20_case_table_fingerprint": case_table_fingerprint,
    "final_family_counts": family_counts,
    "n_release_candidates": int(len(release_candidates)),
    "n_restricted_working_hits": int(len(restricted_working)),
    "stage_d_gates": {str(number): entry["path"]
                      for number, entry in stage_d_gates.items()},
}
sd.write_json_atomic(NB22_DIR / "gate_nb22.json", gate_payload)

# Write the manifest last so it hashes NB 22's final gate and configuration. If an expected
# NB 22 product is absent, rewrite the gate before taking the authoritative snapshot.
expected_nb22_outputs = {
    "stage_D/nb22_release/environment.json",
    "stage_D/nb22_release/model_references.csv",
    "stage_D/nb22_release/reference_audit.csv",
    "stage_D/nb22_release/restricted_data_hits.csv",
    "stage_D/nb22_release/restricted_working_hits.csv",
    "stage_D/nb22_release/release_allowlist.csv",
    "stage_D/nb22_release/protocol_deviations.csv",
    "stage_D/nb22_release/protocol_deviations.md",
    "stage_D/nb22_release/release_checklist.csv",
    "stage_D/nb22_release/release_checklist.md",
    "stage_D/nb22_release/run_config.json",
    "stage_D/nb22_release/gate_nb22.json",
}
manifest = build_manifest()
manifest_paths = set(manifest["path"].astype(str))
missing_nb22_outputs = sorted(expected_nb22_outputs - manifest_paths)
if missing_nb22_outputs:
    message = f"NB 22 did not produce its managed outputs: {missing_nb22_outputs}."
    hard_failures.append(message)
    analysis_passed = False
    automated_release_ready = False
    blocking_failures = list(hard_failures)
    if ENFORCE_CAMERA_READY:
        blocking_failures.extend(release_blockers)
    gate_payload.update({
        "passed": False,
        "analysis_passed": False,
        "automated_release_ready": False,
        "failures": blocking_failures,
        "computational_failures": hard_failures})
    sd.write_json_atomic(NB22_DIR / "gate_nb22.json", gate_payload)
    manifest = build_manifest()
manifest.to_csv(NB22_DIR / "artifact_manifest.csv", index=False)
print(f"Final artifact manifest: {len(manifest):,} files")

if blocking_failures:
    detail = "\n".join(f"  [{index + 1}] {message}"
                       for index, message in enumerate(blocking_failures))
    label = "camera-ready" if ENFORCE_CAMERA_READY else "analysis"
    raise AssertionError(
        f"NB 22 {label} gate failed with {len(blocking_failures)} blocking issue(s):\n{detail}")

print()
print("NB 22 analysis gate: PASSED")
if automated_release_ready:
    print("Automated camera-ready checks: READY")
else:
    print(f"Automated camera-ready checks: NOT READY ({len(release_blockers)} blocker(s)).")
    print("Resolve the items listed above, set ENFORCE_CAMERA_READY = True, and rerun NB 22 "
          "before submission.")
print(f"Manual author sign-offs still listed: {len(manual_items)}")
print()
print(f"Release checklist: {int((checklist['Status'] == 'READY').sum())} ready, "
      f"{len(blocked_items)} blocked, {int((checklist['Status'] == 'MANUAL').sum())} manual.")
print(f"See {NB22_DIR / 'release_checklist.md'}")
